# A Shift/Reduce Parser with Parse Trees and Abstract Syntax

F# counterpart of `ANTLR4-Python/SLR-Parser-Generator/Shift-Reduce-Parser-AST.ipynb`. All original exercises have concrete implementations. The source repository omitted this notebook’s parse table, tree helper and sum-for fixture. This edition supplies the fixture, generates an SLR table from `simple.g`, and reuses the canonical typed driver. The grammar’s undefined `literal` reference is corrected to `negation`.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "../../Chapter-08/GrammarSyntax/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "../../Chapter-08/GrammarSyntax/bin/Debug/net9.0/GrammarSyntax.dll"
#load "../../Chapter-09/SLR.fsx"
#load "../../Chapter-09/ParseTable.fsx"
#load "../../Chapter-09/ShiftReducePure.fsx"
#load "../../Support/Visualization.fsx"
open GrammarSyntax
open SLR
open ShiftReducePure
open System
open System.IO
open System.Text.RegularExpressions

In [ ]:
let grammar = GrammarSyntax.Parse.file "simple.g"
let table = SLR.computeTables grammar
checkNotebook (SLR.conflicts table |> List.isEmpty)
let terminals = SLR.terminals grammar

## Exercise 1: tokenize

The scanner is anchored at the current position. It converts natural numbers to `NUMBER`, names to `ID`, and grammar keywords to their literal terminal name. `≤` and `>=` forms are normalized to the grammar’s Unicode comparison tokens. Unrecognized input fails at its actual position.

In [ ]:
let tokenizeProgram (source: string) : InputToken list =
    let scanner = Regex(@"\G(?:\s+|//[^\n]*|:=|<=|>=|!=|(?:0|[1-9][0-9]*)|[A-Za-z][A-Za-z0-9_]*|[{}();,+*/=<>≤≥≠∨∧¬-])")
    let rec scan position result =
        if position = source.Length then List.rev result
        else
            let matched = scanner.Match(source, position)
            if not matched.Success then failwithf "Illegal character at offset %d" position
            let text = matched.Value
            let next = position + matched.Length
            if String.IsNullOrWhiteSpace text || text.StartsWith "//" then scan next result
            else
                let normalized = match text with "<=" -> "≤" | ">=" -> "≥" | "!=" -> "≠" | _ -> text
                let kind =
                    if Char.IsDigit text[0] then "NUMBER"
                    elif Set.contains normalized terminals then normalized
                    else "ID"
                scan next ({ Kind = kind; Lexeme = normalized } :: result)
    scan 0 []

## Exercises 2–3: construct parse trees

A leaf keeps both token kind and lexeme. Every reduction keeps its typed production and ordered children. The generic driver owns stack manipulation; the two semantic functions own tree construction.

In [ ]:
type ParseTree = Token of InputToken | Production of Rule * ParseTree list
let parse text =
    parseBy table Token (fun rule children -> Production (rule, children)) (tokenizeProgram text)
    |> Result.defaultWith failwith
let rec drawParseTree tree =
    match tree with
    | Token token -> Visualization.Node (token.Kind, [Visualization.Leaf token.Lexeme])
    | Production (rule, children) -> Visualization.Node (rule.Head, List.map drawParseTree children)

## Exercise 4: combine trees

Punctuation leaves identify an operator or keyword. Other leaves retain meaningful `ID` and `NUMBER` values. A production’s first punctuation token becomes its label; remaining punctuation is discarded.

In [ ]:
type Ast = Atom of string | Labeled of string * Ast list
let rec combineTrees tree =
    match tree with
    | Token token when token.Kind = "ID" || token.Kind = "NUMBER" -> Labeled (token.Kind, [Atom token.Lexeme])
    | Token token -> Atom token.Lexeme
    | Production (_, children) ->
        let combined = List.map combineTrees children
        let label = combined |> List.tryPick (function Atom text -> Some text | _ -> None) |> Option.defaultValue ""
        let subtrees = combined |> List.filter (function Atom _ -> false | _ -> true)
        Labeled (label, subtrees)

## Exercise 5: simplify

Uninformative wrappers collapse, an empty prefix disappears, and remaining unlabeled sequences use `.`. A function-call node takes its callee’s name as its label. These rules are separate from the parser so we can inspect both representations.

In [ ]:
let voidKeys = set [""; "("; ";"; "NUMBER"; "ID"]
let rec simplifyTree tree =
    match tree with
    | Atom _ -> tree
    | Labeled ("", [Labeled ("", []); tail]) -> simplifyTree tail
    | Labeled (label, [child]) when Set.contains label voidKeys -> simplifyTree child
    | Labeled (label, children) ->
        let simplified = List.map simplifyTree children
        match label, simplified with
        | "(", Atom callee :: arguments -> Labeled (callee, arguments)
        | "", _ -> Labeled (".", simplified)
        | _ -> Labeled (label, simplified)
let rec drawAst tree =
    match tree with
    | Atom text -> Visualization.Leaf text
    | Labeled (label, children) -> Visualization.Node (label, List.map drawAst children)
let rec contains label tree =
    match tree with
    | Atom _ -> false
    | Labeled (head, children) -> head = label || List.exists (contains label) children

In [ ]:
let source = File.ReadAllText "sum-for.sl"
printfn "%s" source
let concrete = parse source
let abstractTree = concrete |> combineTrees |> simplifyTree
checkNotebook (contains "function" abstractTree && contains "for" abstractTree && contains "return" abstractTree)
printfn "%A\n%A" concrete abstractTree
let invalid = parseBy table Token (fun rule children -> Production (rule, children)) (tokenizeProgram "x := ;")
checkNotebook (Result.isError invalid)
let invalidNumber = try parse "x := 007;" |> ignore; false with _ -> true
checkNotebook invalidNumber


In [ ]:
let show tree = Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(Visualization.treeToDot tree |> Visualization.renderDot), "text/html") |> ignore
show (drawParseTree concrete)
show (drawAst abstractTree)